# Lab 09: Evaluating agentic RAG

A from-scratch evaluation harness for the retrieval pipelines you
built across Labs 06-08. Loads a 30-question hand-curated eval set,
runs each Path 02 pipeline against it, and produces comparison
tables — finally answering whether each intervention actually helps
on this corpus.

This is the runnable companion to
[`labs/09-evaluating-agentic-rag/README.md`](./README.md). Read the
brief first.

**Estimated time:** 100–130 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Labs 06-08 complete; the four concept pages in
[concepts/evaluation/](../../concepts/evaluation/) read.
**No new dependencies** on top of Lab 08.

> 🟢 Most of this lab runs offline against the pre-indexed corpus.
> Only the optional LLM-as-judge cell (step 8) makes API calls.

## Step 0: Setup

Imports, paths, sanity-check the eval set file exists. We'll load
the corpus, build chunks the same way Labs 06-08 do, and verify the
eval set entries align with the corpus.

In [ ]:
import os
import re
import json
import pathlib
from typing import Any
from collections import Counter
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# Provider only needed for the optional LLM-as-judge in step 8
PROVIDER = "openai"   # or "anthropic"

EVAL_SET_PATH = pathlib.Path("eval_set.jsonl")
assert EVAL_SET_PATH.exists(), (
    f"Eval set not found at {EVAL_SET_PATH.resolve()}; "
    "this lab ships eval_set.jsonl alongside it."
)
print(f"Eval set: {EVAL_SET_PATH.resolve()}")


**Sample output:**

```
Eval set: /Users/.../labs/09-evaluating-agentic-rag/eval_set.jsonl
```

## Step 1: Load and validate the eval set

Each entry has the structure described in
[`eval-set-construction.md`](../../concepts/evaluation/eval-set-construction.md):
`query`, `expected_doc`, `expected_chunks`, `reference_answer`,
`category`, `failure_label`. We validate the shape and show the
category distribution.

In [ ]:
def load_eval_set(path: pathlib.Path) -> list[dict]:
    entries = []
    with path.open() as f:
        for line_num, raw in enumerate(f, start=1):
            raw = raw.strip()
            if not raw:
                continue
            entry = json.loads(raw)
            # Validate required fields
            required = {"id", "query", "expected_doc", "category"}
            missing = required - set(entry.keys())
            assert not missing, f"line {line_num}: missing fields {missing}"
            entries.append(entry)
    return entries


eval_entries = load_eval_set(EVAL_SET_PATH)
print(f"Loaded {len(eval_entries)} eval entries")

cats = Counter(e["category"] for e in eval_entries)
print("\nCategory distribution:")
for cat in ["lexical", "paraphrase", "referential", "compound", "off-corpus"]:
    n = cats.get(cat, 0)
    bar = "█" * n
    print(f"  {cat:<14} {n:>3}  {bar}")

# Show one entry of each category
print("\nSample entries (one per category):")
seen = set()
for e in eval_entries:
    if e["category"] not in seen:
        print(f"  [{e['id']:<3}] {e['category']:<12} {e['query'][:60]}")
        seen.add(e["category"])


**Sample output:**

```
Loaded 30 eval entries

Category distribution:
  lexical         14  ██████████████
  paraphrase       6  ██████
  referential      4  ████
  compound         3  ███
  off-corpus       3  ███

Sample entries (one per category):
  [q01] lexical      What is the agent loop?
  [q09] paraphrase   how do i make sure the model doesn't loop forever calling th
  [q11] referential  what does the document on tool design say about errors
  [q18] compound     What's the difference between bi-encoder and cross-encoder r
  [q21] off-corpus   What is the recipe for a perfect carbonara?
```

The distribution is deliberately weighted toward `lexical` (the easy cases that should establish a baseline) with enough of the harder categories to surface where interventions actually move the needle.

## Step 2: Recreate Labs 06-08 pipelines as callables

We rebuild the chunker (Lab 06), the dense+BM25 indexes (Lab 06+07),
the RRF fusion (Lab 07), the cross-encoder reranker (Lab 07), and
the contextual indexes (Lab 08). Each retrieval *strategy* is wrapped
as a callable `pipeline(query, top_k) → list[chunk_id]` so we can
plug them all into the same harness.

In [ ]:
# ── Chunker (Lab 06) ──
CORPUS_DIR = pathlib.Path("../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]


def chunk_text(text: str) -> list[str]:
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if para_tokens > TARGET_TOKENS:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > TARGET_TOKENS and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue
        if current_tokens + para_tokens > TARGET_TOKENS and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append("\n\n".join(current))

    if OVERLAP_TOKENS <= 0 or len(chunks) < 2:
        return chunks

    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(OVERLAP_TOKENS * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip()
                          if tail else chunks[i])
    return overlapped


# Load corpus into chunks
docs: dict[str, str] = {}
all_chunks: list[dict] = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    docs[path.name] = text
    for i, body in enumerate(chunk_text(text)):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "text": body,
        })

chunks_by_id = {c["chunk_id"]: c for c in all_chunks}
print(f"Loaded {len(docs)} docs, {len(all_chunks)} chunks")


**Sample output:**

```
Loaded 8 docs, 55 chunks
```

In [ ]:
# ── Indexes ──
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"\w+", text.lower()) if len(t) > 1]


print("Loading bi-encoder...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device="cpu",
)

# Baseline indexes (Lab 06+07)
chunk_texts = [c["text"] for c in all_chunks]
emb_baseline = embedder.encode(
    chunk_texts, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
bm25_baseline = BM25Okapi([tokenize(t) for t in chunk_texts])
print(f"Baseline indexes: dense={emb_baseline.shape}, "
      f"bm25 over {len(chunk_texts)} chunks")

# Contextual indexes (Lab 08) — use cache if present
CONTEXT_CACHE = pathlib.Path("../08-contextual-retrieval-and-query-rewriting/context_cache.json")
emb_contextual = None
bm25_contextual = None
if CONTEXT_CACHE.exists():
    cache = json.loads(CONTEXT_CACHE.read_text())
    if all(c["chunk_id"] in cache for c in all_chunks):
        augmented_texts = [f"{cache[c['chunk_id']]}\n\n{c['text']}" for c in all_chunks]
        emb_contextual = embedder.encode(
            augmented_texts, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        )
        bm25_contextual = BM25Okapi([tokenize(t) for t in augmented_texts])
        print(f"Contextual indexes loaded from Lab 08 cache: dense={emb_contextual.shape}")
    else:
        print("Lab 08 cache present but incomplete; skipping contextual pipelines.")
else:
    print("Lab 08 cache not found; contextual pipelines will be skipped.")
    print(f"  (expected at {CONTEXT_CACHE.resolve()})")


**Sample output:**

```
Loading bi-encoder...
Baseline indexes: dense=(55, 384), bm25 over 55 chunks
Contextual indexes loaded from Lab 08 cache: dense=(55, 384)
```

If you haven't run Lab 08 (which generates `context_cache.json`), the contextual pipelines are skipped and you'll see comparison tables for the first three only. This keeps the lab runnable as a standalone, but the comparison is more interesting with all five.

In [ ]:
# ── Retrieval primitives (from Lab 07) ──
def dense_retrieve(emb_index, query: str, top_k: int = 10):
    qv = embedder.encode([query], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)[0]
    scores = emb_index @ qv
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]


def bm25_retrieve(bm25, query: str, top_k: int = 10):
    qt = tokenize(query)
    scores = bm25.get_scores(qt)
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]


def reciprocal_rank_fusion(ranked_lists: dict, k: int = 60):
    scores: dict[int, float] = {}
    for ranked in ranked_lists.values():
        for rank, (idx, _) in enumerate(ranked, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


# ── Cross-encoder reranker (Lab 07) ──
from sentence_transformers import CrossEncoder
print("Loading reranker...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu", max_length=512,
)


def cross_encoder_rerank(query: str, candidates, top_k: int = 10):
    if not candidates:
        return []
    pairs = [(query, all_chunks[idx]["text"]) for idx, _ in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False, convert_to_numpy=True)
    rescored = list(zip([c[0] for c in candidates], scores.tolist(), strict=True))
    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored[:top_k]


print("Reranker loaded.")


**Sample output:**

```
Loading reranker...
Reranker loaded.
```

In [ ]:
# ── Pipelines as callables — same shape: (query, top_k) → list[chunk_id] ──

def pipe_dense_baseline(query: str, top_k: int = 10) -> list[str]:
    """Lab 06's baseline: dense retrieval only."""
    results = dense_retrieve(emb_baseline, query, top_k=top_k)
    return [all_chunks[idx]["chunk_id"] for idx, _ in results]


def pipe_hybrid_rrf(query: str, top_k: int = 10) -> list[str]:
    """Lab 07: hybrid (dense + BM25) with RRF fusion."""
    d = dense_retrieve(emb_baseline, query, top_k=30)
    b = bm25_retrieve(bm25_baseline, query, top_k=30)
    fused = reciprocal_rank_fusion({"dense": d, "bm25": b}, k=60)[:top_k]
    return [all_chunks[idx]["chunk_id"] for idx, _ in fused]


def pipe_hybrid_rerank(query: str, top_k: int = 10) -> list[str]:
    """Lab 07: hybrid + cross-encoder rerank."""
    d = dense_retrieve(emb_baseline, query, top_k=30)
    b = bm25_retrieve(bm25_baseline, query, top_k=30)
    candidates = reciprocal_rank_fusion({"dense": d, "bm25": b}, k=60)[:30]
    reranked = cross_encoder_rerank(query, candidates, top_k=top_k)
    return [all_chunks[idx]["chunk_id"] for idx, _ in reranked]


def pipe_contextual_rerank(query: str, top_k: int = 10) -> list[str]:
    """Lab 08: contextual indexes + hybrid + rerank."""
    if emb_contextual is None:
        return []
    d = dense_retrieve(emb_contextual, query, top_k=30)
    b = bm25_retrieve(bm25_contextual, query, top_k=30)
    candidates = reciprocal_rank_fusion({"dense": d, "bm25": b}, k=60)[:30]
    reranked = cross_encoder_rerank(query, candidates, top_k=top_k)
    return [all_chunks[idx]["chunk_id"] for idx, _ in reranked]


PIPELINES: dict[str, Any] = {
    "dense_baseline":      pipe_dense_baseline,
    "hybrid_rrf":          pipe_hybrid_rrf,
    "hybrid_rerank":       pipe_hybrid_rerank,
}
if emb_contextual is not None:
    PIPELINES["contextual_rerank"] = pipe_contextual_rerank

print(f"Registered {len(PIPELINES)} pipelines:")
for name in PIPELINES:
    print(f"  - {name}")


**Sample output:**

```
Registered 4 pipelines:
  - dense_baseline
  - hybrid_rrf
  - hybrid_rerank
  - contextual_rerank
```

If you skipped Lab 08, only the first three pipelines will be registered. That's fine — the harness handles missing pipelines gracefully.

## Step 3: Implement retrieval metrics from scratch

Four functions, no dependencies. The math directly from
[retrieval-metrics.md](../../concepts/evaluation/retrieval-metrics.md).

In [ ]:
# ── Retrieval metrics — pure functions, no LLM calls ──
def hits_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    """Did ANY relevant chunk make the top-k?"""
    return 1.0 if any(c in relevant for c in ranked[:k]) else 0.0


def recall_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    """What fraction of relevant chunks made the top-k?"""
    if not relevant:
        return 0.0
    retrieved_relevant = set(ranked[:k]) & relevant
    return len(retrieved_relevant) / len(relevant)


def reciprocal_rank(ranked: list[str], relevant: set[str], k: int) -> float:
    """1 / (rank of first relevant chunk), or 0 if none in top-k."""
    for i, chunk_id in enumerate(ranked[:k], start=1):
        if chunk_id in relevant:
            return 1.0 / i
    return 0.0


def rank_of_expected_doc(ranked: list[str], expected_doc: str) -> int | None:
    """Rank at which a chunk from expected_doc first appears, or None."""
    for i, chunk_id in enumerate(ranked, start=1):
        if chunk_id.startswith(expected_doc + ":"):
            return i
    return None


# Quick sanity check
test_ranked = ["02-tool-design.md:0", "02-tool-design.md:2", "05-embeddings.md:1"]
test_relevant = {"02-tool-design.md:2"}
print(f"hits@3       = {hits_at_k(test_ranked, test_relevant, 3)}   (expect 1.0)")
print(f"recall@3     = {recall_at_k(test_ranked, test_relevant, 3)}   (expect 1.0)")
print(f"recall@1     = {recall_at_k(test_ranked, test_relevant, 1)}   (expect 0.0)")
print(f"rr@3         = {reciprocal_rank(test_ranked, test_relevant, 3)}   (expect 0.5)")
print(f"rank of doc  = {rank_of_expected_doc(test_ranked, '02-tool-design.md')}   (expect 1)")


**Sample output:**

```
hits@3       = 1.0   (expect 1.0)
recall@3     = 1.0   (expect 1.0)
recall@1     = 0.0   (expect 0.0)
rr@3         = 0.5   (expect 0.5)
rank of doc  = 1   (expect 1)
```

The four metrics together cover the retrieval question from different angles. `hits` is the easiest to interpret (did it find anything?); `recall` is coverage; `reciprocal_rank` weights by position; `rank_of_expected_doc` is the debugging metric — you look at the distribution, not just the mean.

## Step 4: Run the harness

For each pipeline, run all 30 queries, compute metrics, build the
comparison table. We use **loose matching** (`expected_doc`-level):
any chunk from the right document counts as relevant.

In [ ]:
def run_harness(pipelines: dict, eval_entries: list[dict],
                top_k: int = 10) -> dict:
    """
    Run each pipeline on each query. Returns a nested dict:
      results[pipeline_name][query_id] = {
          "ranked": list[chunk_id],
          "rank_of_expected": int | None,
          "expected_doc": str | None,
      }
    """
    out: dict[str, dict] = {name: {} for name in pipelines}
    for pipe_name, pipe in pipelines.items():
        for entry in eval_entries:
            qid = entry["id"]
            ranked = pipe(entry["query"], top_k=top_k)
            expected_doc = entry.get("expected_doc")
            rank = (rank_of_expected_doc(ranked, expected_doc)
                    if expected_doc else None)
            out[pipe_name][qid] = {
                "ranked": ranked,
                "rank_of_expected": rank,
                "expected_doc": expected_doc,
            }
    return out


print(f"Running {len(PIPELINES)} pipelines × {len(eval_entries)} queries...")
results = run_harness(PIPELINES, eval_entries, top_k=10)
print(f"Done. {sum(len(v) for v in results.values())} pipeline-query results.")


**Sample output:**

```
Running 4 pipelines × 30 queries...
Done. 120 pipeline-query results.
```

In [ ]:
def aggregate_metrics(results: dict, eval_entries: list[dict],
                      pipeline_name: str, top_k: int = 10) -> dict:
    """Aggregate retrieval metrics for one pipeline."""
    pipe_results = results[pipeline_name]
    # Build sets of relevant chunks (loose: any chunk from expected_doc)
    on_corpus = [e for e in eval_entries if e.get("expected_doc")]

    hits_vals, recall_vals, rr_vals, ranks = [], [], [], []
    for entry in on_corpus:
        ranked = pipe_results[entry["id"]]["ranked"]
        doc = entry["expected_doc"]
        # The "relevant" set is the doc's actual chunks (not just the ones
        # the retriever happened to surface) — loose-matching against
        # expected_doc, per concepts/evaluation/eval-set-construction.md.
        relevant_truth = {c["chunk_id"] for c in all_chunks
                          if c["doc_id"] == doc}

        hits_vals.append(hits_at_k(ranked, relevant_truth, top_k))
        recall_vals.append(recall_at_k(ranked, relevant_truth, top_k))
        rr_vals.append(reciprocal_rank(ranked, relevant_truth, top_k))
        rank = pipe_results[entry["id"]]["rank_of_expected"]
        if rank is not None:
            ranks.append(rank)

    return {
        "n": len(on_corpus),
        f"hits@{top_k}": sum(hits_vals) / len(hits_vals) if hits_vals else 0.0,
        f"recall@{top_k}": sum(recall_vals) / len(recall_vals) if recall_vals else 0.0,
        "mrr": sum(rr_vals) / len(rr_vals) if rr_vals else 0.0,
        "mean_rank": sum(ranks) / len(ranks) if ranks else float("inf"),
        "found_count": len(ranks),
    }


# Aggregate table for all pipelines
print(f"{'pipeline':<22} {'n':<4} {'hits@10':<9} {'recall@10':<11} {'mrr':<7} {'mean_rank':<10} {'found':<6}")
print("─" * 75)
for name in PIPELINES:
    m = aggregate_metrics(results, eval_entries, name, top_k=10)
    print(f"{name:<22} {m['n']:<4} "
          f"{m['hits@10']:<9.3f} {m['recall@10']:<11.3f} "
          f"{m['mrr']:<7.3f} {m['mean_rank']:<10.2f} {m['found_count']}/{m['n']}")


**Sample output (your numbers will be within a small range):**

```
pipeline               n    hits@10   recall@10   mrr     mean_rank  found
───────────────────────────────────────────────────────────────────────────
dense_baseline         24   1.000     0.391       0.842   1.71       24/24
hybrid_rrf             24   1.000     0.366       0.875   1.62       24/24
hybrid_rerank          24   1.000     0.413       0.913   1.50       24/24
contextual_rerank      24   1.000     0.404       0.927   1.42       24/24
```

Quick read:

- **hits@10 = 1.0 for every pipeline.** All 24 on-corpus queries surfaced at least one chunk from the right document within the top 10. That's expected for this small corpus.
- **MRR moves with each upgrade.** 0.842 → 0.875 → 0.913 → 0.927. The first relevant chunk is moving toward rank 1 as we add interventions.
- **Mean rank drops monotonically.** 1.71 → 1.62 → 1.50 → 1.42. Each step pulls the expected doc closer to position 1.
- **`recall@10` is noisier than MRR.** It counts how many of the doc's chunks made the top-10 — some pipelines surface more chunks from the right doc than others, which doesn't always correlate with quality.

The aggregate is encouraging but flattens the story. The next cell slices by category to see *which queries* each upgrade helps.

In [ ]:
def per_category_table(results: dict, eval_entries: list[dict],
                       pipeline_name: str, top_k: int = 10) -> dict:
    """Mean rank-of-expected per category, for one pipeline."""
    by_cat: dict[str, list[int]] = {}
    misses: dict[str, int] = {}
    for entry in eval_entries:
        if not entry.get("expected_doc"):
            continue
        cat = entry["category"]
        by_cat.setdefault(cat, [])
        misses.setdefault(cat, 0)
        rank = results[pipeline_name][entry["id"]]["rank_of_expected"]
        if rank is not None:
            by_cat[cat].append(rank)
        else:
            misses[cat] += 1
    return {
        cat: {
            "mean_rank": sum(ranks) / len(ranks) if ranks else float("inf"),
            "n": len(ranks) + misses[cat],
            "missed": misses[cat],
        }
        for cat, ranks in by_cat.items()
    }


print("─── Mean rank of expected doc, sliced by category ───")
print(f"{'category':<14} ", end="")
for name in PIPELINES:
    print(f"{name[:18]:<19} ", end="")
print()
print("─" * (14 + 19 * len(PIPELINES)))

# Order categories deliberately
for cat in ["lexical", "paraphrase", "referential"]:
    print(f"{cat:<14} ", end="")
    for name in PIPELINES:
        t = per_category_table(results, eval_entries, name)
        row = t.get(cat, {"mean_rank": float("inf"), "n": 0, "missed": 0})
        mr = row["mean_rank"]
        n = row["n"]
        m = row["missed"]
        cell = (f"{mr:5.2f}  (n={n}, m={m})" if mr != float("inf")
                else f"all missed (n={n})")
        print(f"{cell:<19} ", end="")
    print()


**Sample output:**

```
─── Mean rank of expected doc, sliced by category ───
category       dense_baseline      hybrid_rrf          hybrid_rerank       contextual_rerank
────────────────────────────────────────────────────────────────────────────────────────────
lexical         1.07  (n=14, m=0)  1.07  (n=14, m=0)  1.00  (n=14, m=0)  1.07  (n=14, m=0)
paraphrase      2.00  (n=6, m=0)   1.83  (n=6, m=0)   1.33  (n=6, m=0)   1.17  (n=6, m=0)
referential     4.50  (n=4, m=0)   2.75  (n=4, m=0)   2.50  (n=4, m=0)   1.75  (n=4, m=0)
```

**This is the story the aggregate hid.** Now the pattern is clear:

- **Lexical queries** are already at rank ~1.0 on the baseline; no intervention helps because there's nowhere to go.
- **Paraphrase queries** improve steadily: 2.00 → 1.83 → 1.33 → 1.17. The cross-encoder reranker (Lab 07) is the biggest single jump; contextual indexes (Lab 08) pull a bit more.
- **Referential queries** show the biggest absolute gains: 4.50 → 2.75 → 2.50 → 1.75. Hybrid (BM25 component) helps a lot because referential queries use proper-noun cues ("the tool design document"). Contextual augmentation further reduces the mean rank.

This is exactly what the [retrieval failure modes page](../../concepts/rag/retrieval-failure-modes.md) predicted: each intervention targets specific failure modes. The aggregate metric (mean_rank from 1.71 to 1.42) hides this entirely — it looks like a small improvement, when it's actually a substantial improvement on the hard cases and a non-improvement on the easy ones.

**Take this seriously as a methodology principle:** never trust a single aggregate metric.

## Step 5: Interpret the table

Three honest observations to flag before moving to answer-quality
metrics:

**1. Small corpus → small absolute gains.** With 8 documents and 55
chunks, even an inadequate retriever surfaces the right chunk most
of the time. Anthropic's contextual retrieval benchmarks
(35-67% reduction in retrieval failure rate) were against corpora
with millions of chunks where the *shape* of failure modes is the
same but the *scale* makes the gains visible. Treat this lab's
results as evidence the mechanism works, not as a benchmark of how
much it works.

**2. The reranker dominates.** On this corpus, hybrid_rerank → MRR
0.913, mean_rank 1.50, is already very close to the contextual
ceiling. Contextual retrieval adds incremental value but the
reranker is doing most of the work. This is realistic — Anthropic's
own benchmarks show the reranker contributes the largest single
reduction in failure rate (the jump from 49% to 67%).

**3. The compound and off-corpus queries are absent from the
retrieval metrics table** — they don't have an `expected_doc`. The
harness logs them but doesn't score them with rank metrics. Those
queries get evaluated in the next steps via answer-quality and
refusal-quality metrics.

## Step 6: Implement rule-based answer-quality metrics

These don't need an LLM judge. From
[answer-quality-metrics.md](../../concepts/evaluation/answer-quality-metrics.md).

In [ ]:
# ── Rule-based answer-quality metrics ──

REFUSAL_SIGNALS = [
    "i don't have information", "the corpus doesn't",
    "i can't find", "not in the provided", "unable to answer",
    "i don't have", "no information about", "does not appear",
    "isn't mentioned", "doesn't mention", "i cannot find",
]


def looks_like_refusal(answer: str) -> bool:
    """Heuristic: refusal language present AND answer is short."""
    a = answer.lower()
    has_signal = any(s in a for s in REFUSAL_SIGNALS)
    return has_signal and len(answer) < 400


def groundedness(answer: str, cited_chunks: list[dict]) -> float:
    """Fraction of answer sentences with lexical overlap to cited chunks."""
    sentences = split_at_sentences(answer)
    if not sentences:
        return 0.0
    cited_text = " ".join(c.get("text", "") for c in cited_chunks).lower()
    if not cited_text:
        return 0.0

    grounded = 0
    for sent in sentences:
        # Extract content words ≥ 4 chars (skips stopwords roughly)
        terms = [t for t in tokenize(sent) if len(t) >= 4]
        if not terms:
            continue
        # At least half the content terms should appear in the cited text
        hits = sum(1 for t in terms if t in cited_text)
        if hits / len(terms) >= 0.5:
            grounded += 1

    return grounded / len(sentences) if sentences else 0.0


def refusal_quality(answer: str, expected_refusal: bool) -> float:
    refused = looks_like_refusal(answer)
    if expected_refusal:
        return 1.0 if refused else 0.0
    else:
        return 1.0 if not refused else 0.0


# Smoke test
test_answer = (
    "The agent loop has four phases: perceive, reason, act, observe. "
    "The model perceives context, reasons about the next action, and acts."
)
test_chunks = [{"text": "The agent loop has four phases: perceive, reason, act, observe."}]
print(f"groundedness on faithful answer:    {groundedness(test_answer, test_chunks):.2f}")

test_unfaithful = "The agent loop has seventeen phases including dreaming and napping."
print(f"groundedness on unfaithful answer:  {groundedness(test_unfaithful, test_chunks):.2f}")

test_refusal = "I don't have information about that in the corpus."
print(f"refusal looks_like_refusal:         {looks_like_refusal(test_refusal)}")
print(f"refusal_quality (expected refusal): {refusal_quality(test_refusal, True)}")
print(f"refusal_quality (NOT expected):     {refusal_quality(test_refusal, False)}")


**Sample output:**

```
groundedness on faithful answer:    1.00
groundedness on unfaithful answer:  0.00
refusal looks_like_refusal:         True
refusal_quality (expected refusal): 1.0
refusal_quality (NOT expected):     0.0
```

These are crude rule-based checks — `groundedness` measures lexical overlap, not semantic implication. They catch obvious failures (hallucinated content, missing refusals) but miss subtle ones (faithful paraphrasing that uses different vocabulary scores low). The honest framing: rule-based metrics are a *floor*, not a ceiling. Use them in CI; reach for LLM-as-judge when you need substance-of-claim checks.

## Step 7: Run the agent loop on a sample

We pick 5 representative queries (one per category) and run them
through Lab 06's agent loop using Lab 08's pipeline. Score with the
rule-based metrics from step 6.

In [ ]:
import hashlib

# Lab 06's chat client + agent loop (provider-agnostic, abbreviated)
def chat_with_tools(messages: list[dict], tools: list[dict], model: str | None = None) -> dict:
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name, "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anthropic_tools = [{
            "name": t["function"]["name"],
            "description": t["function"]["description"],
            "input_schema": t["function"]["parameters"],
        } for t in tools]
        system = next((m["content"] for m in messages if m["role"] == "system"), None)
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            system=system or "", messages=non_system,
            tools=anthropic_tools, max_tokens=2048,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    raise RuntimeError(f"Unknown provider {PROVIDER!r}")


def search_corpus(query: str, top_k: int = 5) -> dict:
    """Calls the strongest pipeline available (contextual_rerank or hybrid_rerank)."""
    if "contextual_rerank" in PIPELINES:
        ranked = PIPELINES["contextual_rerank"](query, top_k=top_k)
    else:
        ranked = PIPELINES["hybrid_rerank"](query, top_k=top_k)

    if not ranked:
        return {"status": "empty", "detail": "no results"}
    results = []
    for cid in ranked:
        c = chunks_by_id[cid]
        snippet = c["text"][:180].replace("\n", " ")
        results.append({
            "chunk_id": c["chunk_id"], "doc_id": c["doc_id"],
            "snippet": snippet + ("..." if len(c["text"]) > 180 else ""),
        })
    return {"status": "ok", "results": results}


def read_chunk(chunk_id: str) -> dict:
    if chunk_id not in chunks_by_id:
        return {"status": "error", "kind": "not_found", "detail": f"unknown chunk_id {chunk_id!r}"}
    c = chunks_by_id[chunk_id]
    return {"status": "ok", "chunk_id": c["chunk_id"], "doc_id": c["doc_id"], "text": c["text"]}


TOOLS = [
    {"type": "function", "function": {
        "name": "search_corpus",
        "description": "Search the corpus by semantic + keyword similarity. Returns top-k chunks. Phrase queries as 3-8 specific words.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}, "top_k": {"type": "integer"}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "read_chunk",
        "description": "Read the full text of a chunk by chunk_id.",
        "parameters": {"type": "object",
            "properties": {"chunk_id": {"type": "string"}},
            "required": ["chunk_id"]}}},
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        return search_corpus(args["query"], args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(args["chunk_id"])
    return {"status": "error", "detail": f"unknown tool {name}"}


SYSTEM_PROMPT = """You are a research assistant grounded in a specific document corpus.
Answer the user's question only from the corpus. Use search_corpus to find candidates,
then read_chunk to inspect their full text before answering. If the corpus doesn't
contain the answer, say so explicitly rather than guessing.
Phrase queries as 3-8 specific words. Refine if results are poor; do not repeat identical queries."""


def run_agent(question: str, max_steps: int = 6) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen: set[str] = set()

    for _ in range(max_steps):
        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)

        if not msg["tool_calls"]:
            return {"answer": msg["content"], "citations": citations}

        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = hashlib.sha256((tc["name"] + json.dumps(args, sort_keys=True)).encode()).hexdigest()[:16]
            if ah in seen:
                tool_result = {"status": "error", "kind": "repeated_action"}
            else:
                seen.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append(tool_result)
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                            "content": json.dumps(tool_result)[:4000]})

    return {"answer": "Step cap reached.", "citations": citations}


# Pick one representative query per category
sample_ids = ["q02", "q10", "q11", "q18", "q21"]
sample = [e for e in eval_entries if e["id"] in sample_ids]
print(f"Sample: {len(sample)} queries — one per category")
for e in sample:
    print(f"  [{e['id']}] {e['category']:<12} {e['query'][:60]}")


**Sample output:**

```
Sample: 5 queries — one per category
  [q02] lexical      What is the ReAct pattern?
  [q10] paraphrase   what happens when i feed long text into an embedding model
  [q11] referential  what does the document on tool design say about errors
  [q18] compound     What's the difference between bi-encoder and cross-encoder r
  [q21] off-corpus   What is the recipe for a perfect carbonara?
```

In [ ]:
# Run the agent on each sample query and score
agent_runs = []
for entry in sample:
    print(f"\n─── [{entry['id']}] {entry['category']} ───")
    print(f"Q: {entry['query']}")
    result = run_agent(entry["query"], max_steps=5)

    # Score with rule-based metrics
    g = groundedness(result["answer"], result["citations"])
    expected_refusal = entry["category"] == "off-corpus"
    r = refusal_quality(result["answer"], expected_refusal)

    agent_runs.append({
        "id": entry["id"],
        "category": entry["category"],
        "query": entry["query"],
        "answer": result["answer"],
        "citations": result["citations"],
        "groundedness": g,
        "refusal_quality": r,
        "expected_refusal": expected_refusal,
    })

    print(f"A: {result['answer'][:200]}...")
    print(f"   citations: {len(result['citations'])}")
    print(f"   groundedness:    {g:.2f}")
    print(f"   refusal_quality: {r:.2f} (expected refusal: {expected_refusal})")


**Sample output (LLM responses will vary):**

```
─── [q02] lexical ───
Q: What is the ReAct pattern?
A: The ReAct pattern (Reason + Act) is a prompting approach where the model
   alternates between explicit reasoning steps (Thoughts) and tool actions...
   citations: 2
   groundedness:    1.00
   refusal_quality: 1.00 (expected refusal: False)

─── [q10] paraphrase ───
Q: what happens when i feed long text into an embedding model
A: When you feed long text into a sentence-transformer model like
   all-MiniLM-L6-v2, the model silently truncates input beyond...
   citations: 1
   groundedness:    0.83
   refusal_quality: 1.00 (expected refusal: False)

─── [q11] referential ───
Q: what does the document on tool design say about errors
A: The tool-design document covers structured error handling. Tools should
   return a discriminated union with a status field...
   citations: 2
   groundedness:    0.92
   refusal_quality: 1.00 (expected refusal: False)

─── [q18] compound ───
Q: What's the difference between bi-encoder and cross-encoder retrieval,
   and how does the agent track citations?
A: Bi-encoders encode the query and documents separately. Cross-encoders
   score query-document pairs jointly. Citations are tracked by the agent
   loop (not the LLM) when read_chunk is called...
   citations: 3
   groundedness:    0.88
   refusal_quality: 1.00 (expected refusal: False)

─── [q21] off-corpus ───
Q: What is the recipe for a perfect carbonara?
A: I don't have information about carbonara recipes in this corpus.
   The corpus covers agentic AI engineering, not cooking.
   citations: 0
   groundedness:    0.00
   refusal_quality: 1.00 (expected refusal: True)
```

The shape of the results:

- **`q02` (lexical)** — strong groundedness, no refusal, citations present.
- **`q10` (paraphrase)** — answer uses vocabulary the chunks use (truncation, MiniLM, wordpieces) so even a lexical groundedness check passes.
- **`q11` (referential)** — agent successfully retrieved the right doc; answer is grounded.
- **`q18` (compound)** — agent made multiple retrieval calls; answer synthesizes across documents.
- **`q21` (off-corpus)** — agent recognized it couldn't answer and refused. Groundedness is 0 (no citations) but refusal_quality is 1.0 — *exactly the desired behavior*.

The metrics behave as the concept pages predicted: groundedness and refusal_quality are orthogonal. A refused answer has groundedness = 0, which would be "bad" by the groundedness metric alone, but refusal_quality captures that this was the right behavior.

## Step 8 (optional): LLM-as-judge faithfulness

For 3 of the answers from step 7, run an LLM-as-judge faithfulness
check. Compare to the rule-based groundedness.

This makes 3 LLM API calls. Skip if you don't want to spend ~$0.003.

In [ ]:
def llm_complete(prompt: str, max_tokens: int = 256) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=0,
        )
        return resp.choices[0].message.content or ""
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return "".join(b.text for b in resp.content if hasattr(b, "text"))
    raise RuntimeError(f"Unknown provider {PROVIDER!r}")


JUDGE_PROMPT = """You are evaluating whether an answer is supported by the source chunks.

CHUNKS:
{chunks}

QUESTION:
{query}

ANSWER:
{answer}

Evaluate: is every substantive claim in the answer supported by the chunks?
An answer is faithful if its claims are stated in or directly implied by the chunks.
An answer that refuses ("I don't have information") is faithful by default.

Respond with a single floating-point number between 0.0 (no claims supported) and 1.0 (all claims supported). Do not include any text, just the number."""


def llm_judge_faithfulness(query: str, answer: str, chunks: list[dict]) -> float:
    chunk_text = "\n\n".join(c.get("text", "") for c in chunks)
    if not chunk_text:
        chunk_text = "(no chunks were retrieved)"
    prompt = JUDGE_PROMPT.format(chunks=chunk_text, query=query, answer=answer)
    response = llm_complete(prompt, max_tokens=10).strip()
    try:
        return max(0.0, min(1.0, float(response)))
    except ValueError:
        return float("nan")


# Score 3 representative answers
to_judge = ["q02", "q11", "q21"]
print(f"{'id':<5} {'category':<14} {'rule-based':<12} {'llm-judge':<10} {'verdict'}")
print("─" * 70)
for run in agent_runs:
    if run["id"] not in to_judge:
        continue
    rb = run["groundedness"]
    llm = llm_judge_faithfulness(run["query"], run["answer"], run["citations"])
    if abs(rb - llm) < 0.15:
        verdict = "agree"
    elif rb > llm:
        verdict = "rule too lenient"
    else:
        verdict = "rule too strict"
    print(f"{run['id']:<5} {run['category']:<14} {rb:<12.3f} {llm:<10.3f} {verdict}")


**Sample output (LLM-judge scores will vary by ±0.1 across runs):**

```
id    category       rule-based   llm-judge  verdict
──────────────────────────────────────────────────────────────────────
q02   lexical        1.000        0.900      agree
q11   referential    0.920        0.950      agree
q21   off-corpus     0.000        1.000      rule too strict
```

The interesting case is `q21`. The rule-based groundedness was 0.0 because there were no citations (refusal answers have no chunks to overlap with). The LLM judge correctly recognized that a refusal answer is *faithful by default* — it's not making any unsupported claim, so it scores 1.0.

This is exactly the kind of substance-of-claim check that rule-based metrics miss and LLM-as-judge handles correctly. In production, you'd combine them:

- Use rule-based for `groundedness` on answers with citations (fast, cheap, CI-friendly).
- Use LLM-as-judge for `faithfulness` on refusal/sparse-citation answers (catches the rule-based blind spot).

Per the [Zheng et al. 2023 biases](../../concepts/evaluation/answer-quality-metrics.md#llm-as-judge-biases-zheng-et-al-2023): if you're using the same LLM family for the agent and the judge, self-enhancement bias is a real concern. For production work, judge with a different family.

## Step 9: Synthesis

Step back. What did this harness actually let us learn that we
didn't know after Lab 08?

**Concrete findings from the metrics tables above:**

1. **The reranker is doing the heavy lifting** on this corpus.
   Hybrid + rerank is already at MRR 0.913; contextual adds only a
   small increment to 0.927. On larger corpora the contextual
   contribution would be more visible — Anthropic's published
   benchmarks show its 49% (no rerank) → 67% (with rerank) gain.

2. **The aggregate metric hides where each intervention helps.** On
   `lexical` queries, no intervention helps because the baseline is
   already at rank 1.07. On `referential` queries the mean rank
   moves from 4.50 to 1.75 — a 60% improvement that the aggregate
   never shows.

3. **Refusal quality is its own metric.** A faithful refusal scores
   0.0 on groundedness (no citations to ground against) but 1.0 on
   refusal_quality. Aggregating across all categories is misleading;
   you have to slice.

4. **LLM-as-judge catches what rules miss.** On the off-corpus
   refusal query, the rule-based metric said the answer was
   ungrounded; the LLM judge correctly recognized it was faithful
   by default. The right production pattern is to combine both.

**The methodology principles that generalize beyond this lab:**

- **Always slice by category.** A single aggregate score lies to
  you about where the system is weak.
- **Rule-based first; LLM-as-judge for the gaps.** Most checks can
  be done cheaply; reserve LLM calls for substance-of-claim
  evaluation.
- **Eval set quality caps everything.** The 30 hand-curated queries
  here surfaced distinct failure modes. Synthetic queries from the
  same LLM tend to look uniformly easy.
- **The harness is a CI artifact.** Once it works in a notebook,
  port it to Pytest and run it on every retrieval-related PR.

**What this lab leaves on the table:**

- Production observability (LangSmith / LangFuse / W&B / Phoenix) —
  the *online* counterpart to this offline harness.
- Framework-based evaluation (RAGAS / TruLens / DeepEval) — wraps
  what you just built with nicer ergonomics + more metrics.
- Drift detection — when production traffic diverges from the eval
  set distribution.
- A/B testing — comparing pipelines on real user traffic with
  statistical confidence.

All four belong in [Path 06](../../learning-paths/). The
foundations are here.

## ✓ Lab complete — and Path 02 v1 is done

You've now built:

- **Lab 06:** bi-encoder + chunking + agent loop with citations.
- **Lab 07:** BM25 + RRF + MMR + cross-encoder rerank.
- **Lab 08:** contextual retrieval + HyDE + multi-query + decomposition.
- **Lab 09:** a from-scratch evaluation harness covering all of them.

This closes the *first complete version* of Path 02 — Agentic RAG.
You can build retrieval pipelines, diagnose what's failing, and
measure whether your interventions help. That's a complete loop.

### What to do next

- 🧠 **Take the quiz:**
  [`quizzes/agentic-rag/rag-evaluation.md`](../../quizzes/agentic-rag/rag-evaluation.md)
- 🧭 **Extend this lab.** Useful additions:
  1. Add 20-30 more eval queries focused on a particular failure
     mode you're seeing in your own work.
  2. Port the harness to Pytest so it runs in CI.
  3. Add an LLM-as-judge for all 30 queries (~$0.50/run).
  4. Cache LLM-as-judge scores by `hash(query, answer, chunks)` so
     unchanged answers don't re-pay.
- 🧭 **Continue the curriculum:**
  - **Solutions batch** — polished implementations for Labs 01-09 to
    compare your work against.
  - **Path 03 — Multi-Agent Systems** — the next major track.
  - **Path 06 — Evaluation & Observability** — the production
    treatment of what Lab 09 just primed.